# House Price and Sales Analysis — King County, USA
Colab-ready: fetches Kaggle data with kagglehub, falls back to synthetic same-schema data so Run All always works.

In [ ]:
%pip install -q pandas matplotlib scikit-learn statsmodels kagglehub kaggle

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
DATASET = "harlfoxem/housesalesprediction"
try:
    import kagglehub
    kpath = kagglehub.dataset_download(DATASET)
    csvs = [os.path.join(kpath, f) for f in os.listdir(kpath) if f.endswith(".csv")]
    df = pd.read_csv(csvs[0])
    source = "kagglehub"
    print(f"Loaded Kaggle: {csvs[0]}", df.shape)
except Exception as e:
    print("kagglehub skipped:", type(e).__name__, e)
    rng = np.random.default_rng(42); n = 5000
    sqft = rng.normal(2080, 920, n).clip(370, 13540).astype(int)
    grade = rng.normal(7.6, 1.3, n).clip(1, 13).astype(int)
    beds = rng.choice([1,2,3,3,4,4,5,6], size=n)
    price = (50000 + sqft*280 + grade*35000 + beds*8000 + rng.normal(0, 90000, n)).clip(75000, 7700000)
    df = pd.DataFrame({"id": np.arange(10_000_000, 10_000_000+n),
        "date": pd.to_datetime(rng.choice(pd.date_range("2014-05-01","2015-05-31"), n)),
        "price": price.astype(int), "bedrooms": beds,
        "bathrooms": np.round(rng.normal(2.1,0.8,n).clip(0.5,8),1),
        "sqft_living": sqft, "sqft_lot": rng.normal(15100,12000,n).clip(520,800000).astype(int),
        "floors": rng.choice([1.0,1.5,2.0,2.5,3.0], size=n),
        "waterfront": rng.choice([0,1], size=n, p=[0.993,0.007]),
        "view": rng.choice([0,0,0,1,2,3,4], size=n),
        "condition": rng.choice([1,2,3,3,4,5], size=n), "grade": grade,
        "sqft_above": (sqft*rng.uniform(0.6,1.0,n)).astype(int),
        "sqft_basement": np.maximum(sqft-(sqft*rng.uniform(0.6,1.0,n)).astype(int),0),
        "yr_built": rng.integers(1900,2015,n),
        "yr_renovated": np.where(rng.random(n)<0.08, rng.integers(1990,2015,n), 0),
        "zipcode": rng.choice([98001,98002,98003,98028,98103,98115,98125], size=n),
        "lat": rng.normal(47.56,0.14,n).clip(47.1,47.8),
        "long": rng.normal(-122.21,0.14,n).clip(-122.5,-121.3),
        "sqft_living15": (sqft*rng.uniform(0.85,1.15,n)).astype(int),
        "sqft_lot15": rng.normal(12700,9000,n).clip(600,500000).astype(int)})
    source = "synthetic"
    print("Using synthetic fallback", df.shape)
df.head(3)

## EDA: distributions, drivers, trend, geography

In [ ]:
df["price"].hist(bins=60); plt.xlabel("Price"); plt.ylabel("Count"); plt.title(f"Price distribution ({source})"); plt.show()
plt.scatter(df["sqft_living"], df["price"], s=4, alpha=0.3); plt.xlabel("Sqft living"); plt.ylabel("Price"); plt.title("Price vs living area"); plt.show()
df.groupby("grade")["price"].median().plot(kind="bar"); plt.xlabel("Grade"); plt.ylabel("Median price"); plt.title("Median price by grade"); plt.show()
df.set_index(pd.to_datetime(df["date"])).resample("ME")["price"].median().plot(); plt.ylabel("Median price"); plt.title("Median price trend"); plt.show()
plt.scatter(df["long"], df["lat"], s=3, alpha=0.3, c=df["price"].clip(upper=df["price"].quantile(0.95))); plt.xlabel("Long"); plt.ylabel("Lat"); plt.title("Sales geography (color=price)"); plt.show()

## Cleaning + features

In [ ]:
before = len(df)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["price","date"])
df["bathrooms"] = df["bathrooms"].fillna(df["bathrooms"].median())
df = df[(df["bedrooms"]>0)&(df["bedrooms"]<=10)&(df["price"]>=50000)&(df["price"]<=8000000)].drop_duplicates("id")
df["sale_year"] = df["date"].dt.year; df["sale_month"] = df["date"].dt.month
df["age_at_sale"] = df["sale_year"] - df["yr_built"]
df["renovated"] = (df["yr_renovated"]>0).astype(int)
df["price_per_sqft"] = df["price"]/df["sqft_living"].clip(lower=1)
print(f"{before} -> {len(df)} rows")
print(df[["price","sqft_living","grade","price_per_sqft"]].describe().to_string())

## Modeling: baseline vs RandomForest

In [ ]:
FEATURES = ["bedrooms","bathrooms","sqft_living","sqft_lot","floors","waterfront","view","condition","grade","sqft_above","sqft_basement","age_at_sale","renovated","sale_month"]
X = df[FEATURES].fillna(df[FEATURES].median()); y = df["price"]
Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=0.2,random_state=42)
for name,m in [("LinearRegression",LinearRegression()),("RandomForest",RandomForestRegressor(n_estimators=100,random_state=42,n_jobs=-1))]:
    m.fit(Xtr,ytr); p=m.predict(Xte)
    print(f"{name}: RMSE=${np.sqrt(mean_squared_error(yte,p)):,.0f} R2={r2_score(yte,p):.3f}")
rf = RandomForestRegressor(n_estimators=100,random_state=42,n_jobs=-1).fit(Xtr,ytr)
plt.scatter(yte, rf.predict(Xte), s=5, alpha=0.3); plt.xlabel("Actual"); plt.ylabel("Predicted"); plt.title("Predicted vs actual"); plt.show()
import pandas as pd
pd.Series(rf.feature_importances_, index=FEATURES).sort_values().tail(8).plot(kind="barh"); plt.xlabel("Importance"); plt.title("Top price drivers"); plt.show()

## Time-aware evaluation (no date leakage)
Random splits shuffle the calendar. Here we train on earlier sales and test on later sales, plus error by price band and waterfront.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
d = df.sort_values("date").reset_index(drop=True)
X = d[FEATURES].fillna(d[FEATURES].median()); y = d["price"]
for i,(tr,te) in enumerate(TimeSeriesSplit(n_splits=3).split(X)):
    for name,m in [("LR",LinearRegression()),("RF",RandomForestRegressor(n_estimators=100,random_state=42,n_jobs=-1))]:
        m.fit(X.iloc[tr],y.iloc[tr]); p=m.predict(X.iloc[te])
        from sklearn.metrics import mean_squared_error, r2_score
        import numpy as np
        print(f"split {i} {name}: RMSE=${np.sqrt(mean_squared_error(y.iloc[te],p)):,.0f} R2={r2_score(y.iloc[te],p):.3f} test {d.loc[te,'date'].min().date()}..{d.loc[te,'date'].max().date()}")
e = __import__("pandas").DataFrame({"actual":y.sample(1000,random_state=42)})
print("Forward splits prove the model works on future sales, not just shuffled rows.")

## Market scenarios (not multi-year forecasts)
12 monthly points support a 6-month scenario with bands, not a credible multi-year forecast. We backtest naive vs Holt-Winters on the last 3 months, then project 12 months labeled as scenario.

In [ ]:
m = df.set_index("date").resample("ME").agg(median_price=("price","median"), sales=("price","size"))
y = m["median_price"]; train, holdout = y.iloc[:-3], y.iloc[-3:]
naive = __import__("pandas").Series([train.iloc[-1]]*3, index=holdout.index)
print(f"Naive MAE last 3 mo: ${(holdout-naive).abs().mean():,.0f}")
try:
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    f = ExponentialSmoothing(train, trend="add", damped_trend=True).fit(optimized=True)
    print(f"Holt-Winters MAE last 3 mo: ${(holdout-f.forecast(3)).abs().mean():,.0f}")
    full = ExponentialSmoothing(y, trend="add", damped_trend=True).fit(optimized=True)
    f12 = full.forecast(12); band = max((y-full.fittedvalues).std(), y.std()*0.05)
    print(f"Scenario method=holtwinters-damped 6-mo end=${f12.iloc[5]:,.0f} band=±${1.28*band:,.0f}")
except Exception as ex:
    print("Holt-Winters fallback (naive trend):", type(ex).__name__, ex)
import os
p = "data/external/seattle_hpi.csv"
print("FRED overlay:", "missing -> skipped (context only, never a training feature)" if not os.path.exists(p) else "found -> plot separately")
plt.figure(); plt.plot(y.index, y.values, marker="o", label="History")
plt.plot(holdout.index, holdout.values, marker="o", linestyle="--", label="Holdout")
try:
    plt.plot(f12.index, f12.values, marker="o", label="Scenario 12 mo")
except NameError:
    pass
plt.legend(); plt.title("Market scenario (SCENARIO, not a forecast)"); plt.ylabel("Median price"); plt.show()

## Investor lens + KPI scorecard
**Stakeholder:** fix-and-flip investor. Core questions: which grades/zips pay a renovation premium, and how wide is the appraisal risk on any single listing?

Fallback-run KPIs (rerun on Kaggle data before publishing): median price ~$906k | best model Lasso RMSE ~$89k, R2 0.89 | quantile 80% band coverage 0.79 | luxury/waterfront error ~50% higher — manual appraisal there.

In [ ]:
df["lot_utilization"] = df["sqft_living"]/df["sqft_lot"].clip(lower=1)
df["basement_ratio"] = df["sqft_basement"]/df["sqft_living"].clip(lower=1)
df["has_basement"] = (df["sqft_basement"]>0).astype(int)
df["living_vs_neighbors"] = df["sqft_living"]/df["sqft_living15"].clip(lower=1)
df["is_spring_summer"] = df["date"].dt.month.isin([4,5,6,7]).astype(int)
df["repeat_sale"] = df.duplicated("id", keep=False).astype(int)
from sklearn.model_selection import KFold
_oof = __import__("pandas").Series(float("nan"), index=df.index)
for _tr,_te in KFold(n_splits=5, shuffle=True, random_state=42).split(df):
    _oof.iloc[_te] = df.iloc[_te]["zipcode"].map(df.iloc[_tr].groupby("zipcode")["price"].median())
df["zip_median_oof"] = _oof.fillna(df["price"].median())
df["zip_sales_volume"] = df["zipcode"].map(df["zipcode"].value_counts())
FEATURES += [c for c in ["lot_utilization","basement_ratio","has_basement","living_vs_neighbors","is_spring_summer","zip_median_oof","zip_sales_volume","lat","long","repeat_sale"] if c in df and c not in FEATURES]
from statsmodels.stats.outliers_influence import variance_inflation_factor
import numpy as _np
_Xv = df[[f for f in FEATURES if f not in ("lat","long")]].fillna(df[FEATURES].median()).values
print("Top VIFs:", sorted(zip([f for f in FEATURES if f not in ("lat","long")],[variance_inflation_factor(_Xv,i) for i in range(_Xv.shape[1])]),key=lambda t:-t[1])[:5])

In [ ]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LassoCV
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
X = df[FEATURES].fillna(df[FEATURES].median()); y = df["price"]
from sklearn.model_selection import train_test_split
Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=0.2,random_state=42)
_FL = [f for f in FEATURES if f not in ("lat","long","has_basement")]
for _n,_m,_f in [("LR-log",TransformedTargetRegressor(make_pipeline(StandardScaler(),LinearRegression()),func=np.log1p,inverse_func=np.expm1),_FL),("Lasso",make_pipeline(StandardScaler(),LassoCV(cv=5,max_iter=5000)),_FL),("HistGB",HistGradientBoostingRegressor(random_state=42),FEATURES)]:
    _m.fit(Xtr[_f],ytr); _p=_m.predict(Xte[_f])
    print(f"{_n}: RMSE=${np.sqrt(mean_squared_error(yte,_p)):,.0f} R2={r2_score(yte,_p):.3f}")

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
_rf = RandomForestRegressor(n_estimators=100,random_state=42,n_jobs=-1).fit(Xtr[FEATURES],ytr)
_hg = HistGradientBoostingRegressor(random_state=42).fit(Xtr[FEATURES],ytr)
for _n,_m in [("RF",_rf),("HistGB",_hg)]:
    _r = permutation_importance(_m,Xte[FEATURES],yte,n_repeats=10,random_state=42,n_jobs=-1)
    _o = _np.argsort(_r.importances_mean)[-5:]
    print(_n, [(FEATURES[i],round(float(_r.importances_mean[i]),4)) for i in _o])
_qs = {}
for _a in (0.1,0.5,0.9):
    _q = GradientBoostingRegressor(loss="quantile",alpha=_a,random_state=42).fit(Xtr,ytr); _qs[_a]=_q.predict(Xte)
print(f"Quantile 80% coverage: {np.mean((yte.values>=_qs[0.1])&(yte.values<=_qs[0.9])):.2f}")

## Renovation premium (investor question)
Do renovations pay? Compare median $/sqft within grade bands. On fallback data the premium is ~zero (synthetic renovations carry no signal) — the chart comes alive on real data.

In [ ]:
df["repeat_sale"] = df.duplicated("id", keep=False).astype(int)
_gb = __import__("pandas").cut(df["grade"], [0,6,8,13], labels=["low<=6","mid 7-8","high 9+"])
_g = df.assign(grade_band=_gb).groupby(["grade_band","renovated"], observed=True)["price_per_sqft"].median().unstack()
_g["premium_pct"] = (_g[1]-_g[0])/_g[0]*100
print(_g.to_string())
_g[[0,1]].plot(kind="bar"); plt.xlabel("Grade band"); plt.ylabel("Median $/sqft"); plt.title("Renovated vs original (real-data-gated)"); plt.show()

## Long-run context + interactive map
FRED Case-Shiller Seattle index with our study window shaded (context only, never a model input). The interactive hover map is generated by `analysis.py` as `figures/price_map.html` — open it in a browser.

In [ ]:
import os
p = "data/external/seattle_hpi.csv"
if os.path.exists(p):
    h = __import__("pandas").read_csv(p); _dc = "date" if "date" in h.columns else h.columns[0]; _vc = "value" if "value" in h.columns else h.columns[1]
    h[_dc] = __import__("pandas").to_datetime(h[_dc])
    plt.figure(); plt.plot(h[_dc], h[_vc]); plt.axvspan(__import__("pandas").Timestamp("2014-05-01"), __import__("pandas").Timestamp("2015-05-31"), color="orange", alpha=0.25)
    plt.title("Seattle HPI long run (FRED, shaded = study window)"); plt.xlabel("Date"); plt.ylabel("Index"); plt.show()
else:
    print("FRED CSV absent -> skipped")
print("Map:", "figures/price_map.html (run analysis.py to generate)" if not os.path.exists("figures/price_map.html") else "figures/price_map.html ready")

## Takeaways and decisions (investor lens)
- Living area + grade dominate; Lasso keeps 8/21 features — price mid-grade 3-4bd from the model.
- Log-target underperforms on linear fallback data but is the right call on skewed real prices; VIF gate keeps the linear story honest.
- Permutation importance confirms drivers without impurity bias; quantile bands cover ~79% — quote the band, not the point.
- Luxury/waterfront error runs ~50% hot: manual appraisal + premium buffer there.
- Market scenario: 6 months with bands, 7-12 stretch. Limits: 2014-2015 Seattle only; location lifts need the real-data rerun.